# Диагностика модели: регрессия с серийной корреляцией

In [ ]:
import pandas as pd
import numpy as np

from statsmodels.formula.api import ols
from statsmodels.stats.api import durbin_watson, acorr_breusch_godfrey # DW & LM-тесты
from statsmodels.iolib.summary2 import summary_col, summary_params # вывод подгонки

from scipy.stats import chi2, t # chi2&t-распределения и критические значения

# Не показывать Warnings
import warnings
warnings.simplefilter(action='ignore', category=Warning)

Рассмотрим регрессию

$$
\text{наберите уравнение регрессии}
$$

In [ ]:
# импорт данных
df = pd.read_csv('../datasets/Filename.csv')

## Спецификация и подгонка

__Замечание__: сначала подготовим набор данных, т.е. выполним необходимые преобразования данных и добавим новые переменные:

In [ ]:
# добавим новые переменные в набор данных
df

In [ ]:
# спецификация модели через формулу
mod = ols(formula='your specification', data=df)
# подгонка модели с ковариационной матрицей по умолчанию (неробастной)
res = mod.fit()

## DW-тест

Тестируем модель на серийную корреляцию 1-го порядка

In [ ]:
# DW-статистика
durbin_watson(res.resid)

Число наблюдений `.nobs` и число регрессоров модели `.df_model` для нахождения критических значений

In [ ]:
# число наблюдений и число регрессоров
mod.nobs, mod.df_model

__Правила принятия решения по критерию Дарбина-Уотсона__

|Значение $DW$ |Гипотеза $H_0$|Вывод |
|:--:|:--:|:--:|
|$0 \leq DW< d_L$|отвергается|есть положительная автокорреляция|
|$d_L \leq DW \leq d_U$ |?|неопределенность |
|$d_U < DW < 4-d_U$ |не отвергается |автокорреляция отсутствует |
|$4-d_U \leq DW \leq 4-d_L$|?|неопределенность |
|$4-d_L < DW \leq 4$ |отвергается|есть отрицательная автокорреляция|

__По таблице критических значений__ ([`tables/dw-critical-values.md`](../tables/dw-critical-values.md))

$d_L = $

$d_U = $

**ВЫВОД**:

## LM/BG-тест

Тестируем модель на серийную корреляцию *-го порядка

In [ ]:
# LM-тест на автокорреляцию *-го порядка
acorr_order = 
lm, lmpval, fval, fpval = acorr_breusch_godfrey(res, nlags=acorr_order)
lm, lmpval

### Критическое значение $\chi^2$-распределения

In [ ]:
# уровень значимости
sign_level = 
# критическое значение распределения chi2
chi2.isf(q=sign_level, df=acorr_order)

**ВЫВОД**:

## Робастный t-тест (HAC s.e.)

Используем HAC-оценку ковариационной матрицы

In [ ]:
# спецификация модели через формулу
mod = ols(formula='your specification', data=df)
# подгонка модели с ковариационной матрицей по умолчанию (неробастной)
res_ols = mod.fit()
# число лагов для HAC-оценки
L = int(4*(mod.nobs/100)**(2/9))
# подгонка модели с робастной HAC-оценкой ковариационной матрицы
res_hac = mod.fit(cov_type='HAC', cov_kwds={'maxlags': L}, use_t=True)

### Критическое значение t-распределения

In [ ]:
# уровень значимости
sign_level = 
# критическое значение t-распределения
t.isf(q=sign_level/2, df=mod.df_resid)

### Неробастный t-тест на значимость коэффициентов

In [ ]:
# неробастные t-тесты
summary_params(res_ols, alpha=sign_level).round(3)

**ВЫВОД**:

### Робастный t-тест (HAC s.e.) на значимость коэффициентов

In [ ]:
# робастные t-тесты
summary_params(res_hac, alpha=sign_level).round(3)

**ВЫВОД**:

### Результаты двух оцениваний в одной таблице

__Обратим внимание на коды значимости (количество звёздочек у коэффициентов)__

In [ ]:
# неробастные и робастные (HAC) стандартные ошибки
summary_col(results=[res_ols, res_hac], model_names=['OLS-CME', 'HAC-CME'], stars=True)